# Install Libraries

In [1]:
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q langchain-text-splitters
!pip install -q pymupdf pytesseract pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 65.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import fitz
import pytesseract
from PIL import Image

print("PyMuPDF:", fitz.__doc__.split()[1])
print("Tesseract:", pytesseract.get_tesseract_version())

PyMuPDF: 1.28.2:
Tesseract: 4.1.1


In [3]:
pdf_paths = {

    "WHO": 
        "/kaggle/input/datasets/midhulaict/skinova-knowledge-base/Skinova_knowledge_base/WHO/who_common_skin_diseases.pdf",

    "DermNet": 
        "/kaggle/input/datasets/midhulaict/skinova-knowledge-base/Skinova_knowledge_base/Dermnet"
}

In [4]:
import os

pdf_files = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

print("Total PDF files:", len(pdf_files))

for path in pdf_files:
    print(path)

Total PDF files: 8
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/WHO/who_common_skin_diseases.pdf
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_seborrhoeic_keratosis.pdf
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_bcc.pdf
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_dermatofibroma.pdf
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_actinic_keratosis.pdf
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_melanoma.pdf
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_vascular_lesions.pdf
/kaggle/input/datasets/eshamanohar/skinova-knowledge-base/Skinova_knowledge_base/Dermnet/dermnet_melanocytic_naevi.pdf


# Extract text from DermNet File using OCR

In [5]:
import glob
import os

pdf_files = glob.glob("/kaggle/input/**/*.pdf", recursive=True)

print("Total PDF files found:", len(pdf_files))
print("=" * 70)

for i, path in enumerate(pdf_files, 1):
    print(i, os.path.basename(path))

Total PDF files found: 8
1 who_common_skin_diseases.pdf
2 dermnet_seborrhoeic_keratosis.pdf
3 dermnet_bcc.pdf
4 dermnet_dermatofibroma.pdf
5 dermnet_actinic_keratosis.pdf
6 dermnet_melanoma.pdf
7 dermnet_vascular_lesions.pdf
8 dermnet_melanocytic_naevi.pdf


In [6]:
dermnet_files = [
    path for path in pdf_files
    if os.path.basename(path).startswith("dermnet_")
]

print("DermNet PDFs:", len(dermnet_files))
print("=" * 70)

for path in dermnet_files:
    print(os.path.basename(path))

DermNet PDFs: 7
dermnet_seborrhoeic_keratosis.pdf
dermnet_bcc.pdf
dermnet_dermatofibroma.pdf
dermnet_actinic_keratosis.pdf
dermnet_melanoma.pdf
dermnet_vascular_lesions.pdf
dermnet_melanocytic_naevi.pdf


In [7]:
import pymupdf
import pytesseract
from PIL import Image
import io
import os

# Find the melanoma PDF
melanoma_path = next(
    p for p in dermnet_files
    if os.path.basename(p) == "dermnet_melanoma.pdf"
)

# Open PDF
doc = pymupdf.open(melanoma_path)

print("File:", os.path.basename(melanoma_path))
print("Total pages:", len(doc))
print("=" * 70)

# OCR first 2 pages only
for page_num in range(min(2, len(doc))):
    
    page = doc[page_num]
    
    # Render PDF page as image at 2x resolution
    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(2, 2),
        alpha=False
    )
    
    # Convert to PIL image
    image = Image.open(
        io.BytesIO(pix.tobytes("png"))
    )
    
    # OCR
    text = pytesseract.image_to_string(image)
    
    print(f"\n--- PAGE {page_num + 1} ---")
    print(text[:3000])

File: dermnet_melanoma.pdf
Total pages: 14

--- PAGE 1 ---
 

LESIONS (CANCEROUS)

Melanoma

October 2022

Author: Dr Nicole A. Seebacher, Department of Oncology, University of Oxford, United Kingdom, Ux. (2022)
Previous contributors: Dr Amanda Oakley, Dermatologist (1997)
Reviewing dermatologist: Dr lan Coulson

Edited by the DermNet content department

What is melanoma?

Melanoma, also referred to as malignant melanoma, is a potentially very serious skin cancer in which
there is an uncontrolled growth of melanocytes (pigment cells).

Normal melanocytes are found in the basal layer of the epidermis (outer layer of skin). Melanocytes
produce a protein called melanin, which protects skin cells by absorbing ultraviolet (UV) radiation.

Non-cancerous growth of melanocytes results in moles (benign melanocytic naevi) and freckles
(ephelides and lentigines). In contrast, the cancerous growth of melanocytes results in melanoma.
Melanoma is described as:

In situ, if a tumour is confined to th

In [8]:
import pymupdf
import pytesseract
from PIL import Image
import io
import os
import glob

# Folder to save OCR text
ocr_dir = "/kaggle/working/dermnet_ocr"
os.makedirs(ocr_dir, exist_ok=True)

print("Starting OCR...")
print("=" * 70)

for pdf_path in dermnet_files:

    filename = os.path.basename(pdf_path)
    output_name = os.path.splitext(filename)[0] + ".txt"
    output_path = os.path.join(ocr_dir, output_name)

    print(f"\nProcessing: {filename}")

    doc = pymupdf.open(pdf_path)
    all_text = []

    for page_num in range(len(doc)):

        page = doc[page_num]

        # Render page at 2x resolution
        pix = page.get_pixmap(
            matrix=pymupdf.Matrix(2, 2),
            alpha=False
        )

        # Convert to image
        image = Image.open(
            io.BytesIO(pix.tobytes("png"))
        )

        # OCR
        text = pytesseract.image_to_string(image)

        # Store page text
        all_text.append(
            f"\n--- PAGE {page_num + 1} ---\n{text}"
        )

        print(f"  Page {page_num + 1}/{len(doc)} done")

    # Save complete OCR text
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(all_text))

    print(f"Saved: {output_name}")

print("\n" + "=" * 70)
print("OCR COMPLETE")
print("Output folder:", ocr_dir)

Starting OCR...

Processing: dermnet_seborrhoeic_keratosis.pdf
  Page 1/6 done
  Page 2/6 done
  Page 3/6 done
  Page 4/6 done
  Page 5/6 done
  Page 6/6 done
Saved: dermnet_seborrhoeic_keratosis.txt

Processing: dermnet_bcc.pdf
  Page 1/10 done
  Page 2/10 done
  Page 3/10 done
  Page 4/10 done
  Page 5/10 done
  Page 6/10 done
  Page 7/10 done
  Page 8/10 done
  Page 9/10 done
  Page 10/10 done
Saved: dermnet_bcc.txt

Processing: dermnet_dermatofibroma.pdf
  Page 1/3 done
  Page 2/3 done
  Page 3/3 done
Saved: dermnet_dermatofibroma.txt

Processing: dermnet_actinic_keratosis.pdf
  Page 1/6 done
  Page 2/6 done
  Page 3/6 done
  Page 4/6 done
  Page 5/6 done
  Page 6/6 done
Saved: dermnet_actinic_keratosis.txt

Processing: dermnet_melanoma.pdf
  Page 1/14 done
  Page 2/14 done
  Page 3/14 done
  Page 4/14 done
  Page 5/14 done
  Page 6/14 done
  Page 7/14 done
  Page 8/14 done
  Page 9/14 done
  Page 10/14 done
  Page 11/14 done
  Page 12/14 done
  Page 13/14 done
  Page 14/14 done
Sa

# Check OCR Quality

In [9]:
ocr_files = sorted(
    glob.glob("/kaggle/working/dermnet_ocr/*.txt")
)

print("OCR text files:", len(ocr_files))
print("=" * 70)

for path in ocr_files:

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # Basic statistics
    characters = len(text)
    words = len(text.split())

    print(f"{os.path.basename(path)}")
    print(f"  Characters : {characters:,}")
    print(f"  Words      : {words:,}")
    print()

OCR text files: 7
dermnet_actinic_keratosis.txt
  Characters : 9,440
  Words      : 1,470

dermnet_bcc.txt
  Characters : 12,468
  Words      : 1,819

dermnet_dermatofibroma.txt
  Characters : 3,583
  Words      : 515

dermnet_melanocytic_naevi.txt
  Characters : 13,758
  Words      : 2,172

dermnet_melanoma.txt
  Characters : 25,410
  Words      : 3,821

dermnet_seborrhoeic_keratosis.txt
  Characters : 7,204
  Words      : 1,034

dermnet_vascular_lesions.txt
  Characters : 5,320
  Words      : 719



# Extract WHO file

In [10]:
!pip install pypdf -q


In [11]:
import glob
from pypdf import PdfReader
# Find WHO PDF
who_files = [
    path for path in glob.glob("/kaggle/input/**/*.pdf", recursive=True)
    if os.path.basename(path) == "who_common_skin_diseases.pdf"
]

if not who_files:
    raise FileNotFoundError("WHO PDF not found.")

who_path = who_files[0]

print("WHO file:", os.path.basename(who_path))

# Read PDF
reader = PdfReader(who_path)

print("Total pages:", len(reader.pages))
print("=" * 70)

# Extract text
who_text = []

for page_num, page in enumerate(reader.pages, 1):
    text = page.extract_text() or ""
    
    who_text.append(
        f"\n--- PAGE {page_num} ---\n{text}"
    )

    print(f"Page {page_num}/{len(reader.pages)} extracted")

# Combine
who_text = "\n".join(who_text)

# Save
who_output = "/kaggle/working/who_common_skin_diseases.txt"

with open(who_output, "w", encoding="utf-8") as f:
    f.write(who_text)

print("\n" + "=" * 70)
print("WHO EXTRACTION COMPLETE")
print("Characters:", len(who_text))
print("Words:", len(who_text.split()))
print("Saved to:", who_output)

WHO file: who_common_skin_diseases.pdf
Total pages: 45
Page 1/45 extracted
Page 2/45 extracted
Page 3/45 extracted
Page 4/45 extracted
Page 5/45 extracted
Page 6/45 extracted
Page 7/45 extracted
Page 8/45 extracted
Page 9/45 extracted
Page 10/45 extracted
Page 11/45 extracted
Page 12/45 extracted
Page 13/45 extracted
Page 14/45 extracted
Page 15/45 extracted
Page 16/45 extracted
Page 17/45 extracted
Page 18/45 extracted
Page 19/45 extracted
Page 20/45 extracted
Page 21/45 extracted
Page 22/45 extracted
Page 23/45 extracted
Page 24/45 extracted
Page 25/45 extracted
Page 26/45 extracted
Page 27/45 extracted
Page 28/45 extracted
Page 29/45 extracted
Page 30/45 extracted
Page 31/45 extracted
Page 32/45 extracted
Page 33/45 extracted
Page 34/45 extracted
Page 35/45 extracted
Page 36/45 extracted
Page 37/45 extracted
Page 38/45 extracted
Page 39/45 extracted
Page 40/45 extracted
Page 41/45 extracted
Page 42/45 extracted
Page 43/45 extracted
Page 44/45 extracted
Page 45/45 extracted

WHO EXTR

# Inspect the extracted text

In [12]:
print("=" * 80)
print("WHO TEXT SAMPLE")
print("=" * 80)

with open(
    "/kaggle/working/who_common_skin_diseases.txt",
    "r",
    encoding="utf-8"
) as f:
    who_text = f.read()

print(who_text[:5000])


# --------------------------------------------------
# 2. Inspect one DermNet text
# --------------------------------------------------

print("\n\n" + "=" * 80)
print("DERMNET MELANOMA TEXT SAMPLE")
print("=" * 80)

melanoma_txt = "/kaggle/working/dermnet_ocr/dermnet_melanoma.txt"

with open(
    melanoma_txt,
    "r",
    encoding="utf-8"
) as f:
    melanoma_text = f.read()

print(melanoma_text[:5000])

WHO TEXT SAMPLE

--- PAGE 1 ---
 
 
 
 
Common Skin Diseases for Management or Referral 
at Primary Health Care Level 
WHO  guidance with ICD -11 Mortality and Morbidity Statistics (MMS) codes, release 
2026 -01 
Version 1.0 
 
 
For external expert review draft 
 
 
  

--- PAGE 2 ---
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Disclaimer 
This document has been prepared to support the recognition, management and referral of common skin diseases at the 
primary health care (PHC) level. It is intended to assist Ministries of Health, programme managers, educators and primary 
health care providers in strengthening integrated skin health services. 
The classification supports clinical decision-making, training, service delivery and health information systems. It should be 
adapted to national epidemiology, available resources and existing clinical guidelines. 
This document does not replace national clinical guidelines. Countries should adapt its recommend

# Clean and structure the extracted documents

In [13]:
import re

# ============================================================
# CLEAN EXTRACTED TEXT
# ============================================================

WHO_FILE = "/kaggle/working/who_common_skin_diseases.txt"
DERMNET_DIR = "/kaggle/working/dermnet_ocr"
CLEAN_DIR = "/kaggle/working/clean_text"

os.makedirs(CLEAN_DIR, exist_ok=True)


def clean_text(text):
    """
    Clean OCR/PDF extracted text while preserving
    the actual medical content.
    """

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove form-feed characters
    text = text.replace("\f", "\n")

    # Remove repeated whitespace at line ends
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)

    # Fix words broken across lines by hyphenation
    # Example:
    # melan-
    # oma
    # -> melanoma
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove isolated page-number lines such as "1/14", "3/14"
    text = re.sub(r"^\s*\d+\s*/\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # Remove lines containing only a page number
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # Clean excessive spaces
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Final whitespace cleanup
    text = text.strip()

    return text


# ------------------------------------------------------------
# Clean WHO
# ------------------------------------------------------------

with open(WHO_FILE, "r", encoding="utf-8") as f:
    who_text = f.read()

who_clean = clean_text(who_text)

who_output = os.path.join(
    CLEAN_DIR,
    "who_common_skin_diseases_clean.txt"
)

with open(who_output, "w", encoding="utf-8") as f:
    f.write(who_clean)


# ------------------------------------------------------------
# Clean DermNet files
# ------------------------------------------------------------

dermnet_files = sorted(
    f for f in os.listdir(DERMNET_DIR)
    if f.endswith(".txt")
)

for filename in dermnet_files:

    input_path = os.path.join(DERMNET_DIR, filename)

    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    cleaned = clean_text(text)

    output_filename = filename.replace(
        ".txt",
        "_clean.txt"
    )

    output_path = os.path.join(
        CLEAN_DIR,
        output_filename
    )

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(cleaned)


# ------------------------------------------------------------
# Show results
# ------------------------------------------------------------

print("=" * 70)
print("CLEANING COMPLETE")
print("=" * 70)

print(f"Output directory: {CLEAN_DIR}")
print()

for filename in sorted(os.listdir(CLEAN_DIR)):

    path = os.path.join(CLEAN_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"{filename}")
    print(f"  Characters : {len(text):,}")
    print(f"  Words      : {len(text.split()):,}")
    print()

CLEANING COMPLETE
Output directory: /kaggle/working/clean_text

dermnet_actinic_keratosis_clean.txt
  Characters : 9,367
  Words      : 1,464

dermnet_bcc_clean.txt
  Characters : 12,328
  Words      : 1,808

dermnet_dermatofibroma_clean.txt
  Characters : 3,554
  Words      : 513

dermnet_melanocytic_naevi_clean.txt
  Characters : 13,616
  Words      : 2,163

dermnet_melanoma_clean.txt
  Characters : 25,255
  Words      : 3,807

dermnet_seborrhoeic_keratosis_clean.txt
  Characters : 7,142
  Words      : 1,026

dermnet_vascular_lesions_clean.txt
  Characters : 5,242
  Words      : 714

who_common_skin_diseases_clean.txt
  Characters : 101,409
  Words      : 14,220



# Create source metadata

In [14]:
import json

# ============================================================
# CREATE SOURCE METADATA
# ============================================================

CLEAN_DIR = "/kaggle/working/clean_text"
METADATA_FILE = "/kaggle/working/source_metadata.json"

sources = [
    {
        "filename": "who_common_skin_diseases_clean.txt",
        "source": "WHO",
        "source_type": "clinical_guidance",
        "title": "Common Skin Diseases for Management or Referral at Primary Health Care Level",
        "disease": "Common skin diseases",
        "ham10000_class": None
    },
    {
        "filename": "dermnet_melanoma_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Melanoma",
        "disease": "Melanoma",
        "ham10000_class": "MEL"
    },
    {
        "filename": "dermnet_melanocytic_naevi_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Melanocytic naevi",
        "disease": "Melanocytic nevus",
        "ham10000_class": "NV"
    },
    {
        "filename": "dermnet_bcc_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Basal cell carcinoma",
        "disease": "Basal cell carcinoma",
        "ham10000_class": "BCC"
    },
    {
        "filename": "dermnet_actinic_keratosis_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Actinic keratosis",
        "disease": "Actinic keratosis",
        "ham10000_class": "AKIEC"
    },
    {
        "filename": "dermnet_seborrhoeic_keratosis_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Seborrhoeic keratosis",
        "disease": "Benign keratosis",
        "ham10000_class": "BKL"
    },
    {
        "filename": "dermnet_dermatofibroma_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Dermatofibroma",
        "disease": "Dermatofibroma",
        "ham10000_class": "DF"
    },
    {
        "filename": "dermnet_vascular_lesions_clean.txt",
        "source": "DermNet",
        "source_type": "dermatology_reference",
        "title": "Vascular lesions",
        "disease": "Vascular lesions",
        "ham10000_class": "VASC"
    }
]

# Check that every expected file exists
print("Checking source files...\n")

for item in sources:
    path = os.path.join(CLEAN_DIR, item["filename"])

    if os.path.exists(path):
        print("✓", item["filename"])
    else:
        print("✗ MISSING:", item["filename"])

# Save metadata
with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=4, ensure_ascii=False)

print("\n" + "=" * 70)
print("SOURCE METADATA CREATED")
print("=" * 70)

print("Metadata file:")
print(METADATA_FILE)

print("\nTotal sources:", len(sources))

Checking source files...

✓ who_common_skin_diseases_clean.txt
✓ dermnet_melanoma_clean.txt
✓ dermnet_melanocytic_naevi_clean.txt
✓ dermnet_bcc_clean.txt
✓ dermnet_actinic_keratosis_clean.txt
✓ dermnet_seborrhoeic_keratosis_clean.txt
✓ dermnet_dermatofibroma_clean.txt
✓ dermnet_vascular_lesions_clean.txt

SOURCE METADATA CREATED
Metadata file:
/kaggle/working/source_metadata.json

Total sources: 8


# Create RAG chunks

In [15]:
from collections import Counter

# ============================================================
# CREATE RAG CHUNKS
# ============================================================

CLEAN_DIR = "/kaggle/working/clean_text"
METADATA_FILE = "/kaggle/working/source_metadata.json"
CHUNKS_FILE = "/kaggle/working/rag_chunks.json"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150


# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    sources = json.load(f)


# ------------------------------------------------------------
# Normalize text
# ------------------------------------------------------------

def normalize_for_chunking(text):

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove excessive spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


# ------------------------------------------------------------
# Create overlapping chunks
# ------------------------------------------------------------

def create_chunks(text, chunk_size=800, overlap=150):

    words = text.split()

    chunks = []

    start = 0
    chunk_number = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk_words = words[start:end]

        chunk_text = " ".join(chunk_words).strip()

        if chunk_text:

            chunks.append({
                "chunk_number": chunk_number,
                "text": chunk_text,
                "word_count": len(chunk_words)
            })

        # Stop after final chunk
        if end >= len(words):
            break

        # Move forward while retaining overlap
        start = end - overlap

        chunk_number += 1

    return chunks


# ------------------------------------------------------------
# Generate chunks
# ------------------------------------------------------------

all_chunks = []

for source in sources:

    filename = source["filename"]

    path = os.path.join(
        CLEAN_DIR,
        filename
    )

    if not os.path.exists(path):
        print(f"WARNING: Missing file: {filename}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    text = normalize_for_chunking(text)

    source_chunks = create_chunks(
        text,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    for chunk in source_chunks:

        chunk_id = (
            f"{source['source'].lower()}_"
            f"{source['ham10000_class'] or 'GENERAL'}_"
            f"{chunk['chunk_number']:04d}"
        )

        all_chunks.append({

            # Unique identifier
            "chunk_id": chunk_id,

            # Original file
            "filename": filename,

            # Source information
            "source": source["source"],
            "source_type": source["source_type"],
            "title": source["title"],
            "disease": source["disease"],

            # HAM10000 mapping
            "ham10000_class": source["ham10000_class"],

            # Chunk information
            "chunk_number": chunk["chunk_number"],
            "word_count": chunk["word_count"],

            # Actual retrieved text
            "text": chunk["text"]
        })


# ------------------------------------------------------------
# Save chunks
# ------------------------------------------------------------

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:

    json.dump(
        all_chunks,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print("=" * 70)
print("RAG CHUNKING COMPLETE")
print("=" * 70)

print(f"Total chunks : {len(all_chunks):,}")
print(f"Chunk size   : {CHUNK_SIZE} words")
print(f"Overlap      : {CHUNK_OVERLAP} words")


# ------------------------------------------------------------
# Chunks by source
# ------------------------------------------------------------

print("\nChunks by source:")
print("-" * 70)

source_counts = Counter(
    chunk["title"]
    for chunk in all_chunks
)

for source in sources:

    title = source["title"]

    print(
        f"{source['source']:8s} | "
        f"{title:55s} | "
        f"{source_counts[title]:3d}"
    )


# ------------------------------------------------------------
# Word-count statistics
# ------------------------------------------------------------

word_counts = [
    chunk["word_count"]
    for chunk in all_chunks
]

if word_counts:

    print("\nChunk word-count statistics:")
    print("-" * 70)

    print(f"Minimum : {min(word_counts)}")
    print(f"Maximum : {max(word_counts)}")
    print(f"Average : {sum(word_counts) / len(word_counts):.1f}")


# ------------------------------------------------------------
# Verify chunk structure
# ------------------------------------------------------------

print("\nFirst chunk:")
print("-" * 70)

if all_chunks:

    first = all_chunks[0]

    print("chunk_id       :", first["chunk_id"])
    print("filename       :", first["filename"])
    print("source         :", first["source"])
    print("title          :", first["title"])
    print("disease        :", first["disease"])
    print("HAM10000 class :", first["ham10000_class"])
    print("word_count     :", first["word_count"])

    print("\nText preview:")
    print(first["text"][:1000])


print("\nSaved to:")
print(CHUNKS_FILE)

RAG CHUNKING COMPLETE
Total chunks : 42
Chunk size   : 800 words
Overlap      : 150 words

Chunks by source:
----------------------------------------------------------------------
WHO      | Common Skin Diseases for Management or Referral at Primary Health Care Level |  22
DermNet  | Melanoma                                                |   6
DermNet  | Melanocytic naevi                                       |   4
DermNet  | Basal cell carcinoma                                    |   3
DermNet  | Actinic keratosis                                       |   3
DermNet  | Seborrhoeic keratosis                                   |   2
DermNet  | Dermatofibroma                                          |   1
DermNet  | Vascular lesions                                        |   1

Chunk word-count statistics:
----------------------------------------------------------------------
Minimum : 164
Maximum : 800
Average : 733.7

First chunk:
--------------------------------------------------------

# Inspect chunks for retrieval quality

In [16]:
# ============================================================
# INSPECT RAG CHUNKS
# ============================================================

CHUNKS_FILE = "/kaggle/working/rag_chunks.json"

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def show_chunk(chunk):

    print("=" * 80)
    print("CHUNK ID       :", chunk["chunk_id"])
    print("SOURCE         :", chunk["source"])
    print("TITLE          :", chunk["title"])
    print("DISEASE        :", chunk["disease"])
    print("HAM10000 CLASS :", chunk["ham10000_class"])
    print("CHUNK NUMBER   :", chunk["chunk_number"])
    print("WORD COUNT     :", chunk["word_count"])
    print("-" * 80)
    print(chunk["text"][:2500])
    print()


# ------------------------------------------------------------
# 1. First WHO chunk
# ------------------------------------------------------------

print("\n### WHO — FIRST CHUNK ###\n")

who_chunks = [
    c for c in chunks
    if c["source"] == "WHO"
]

show_chunk(who_chunks[0])


# ------------------------------------------------------------
# 2. WHO middle chunk
# ------------------------------------------------------------

print("\n### WHO — MIDDLE CHUNK ###\n")

show_chunk(who_chunks[len(who_chunks) // 2])


# ------------------------------------------------------------
# 3. WHO final chunk
# ------------------------------------------------------------

print("\n### WHO — FINAL CHUNK ###\n")

show_chunk(who_chunks[-1])


# ------------------------------------------------------------
# 4. Melanoma chunk
# ------------------------------------------------------------

print("\n### DERMNET — MELANOMA ###\n")

melanoma_chunks = [
    c for c in chunks
    if c["ham10000_class"] == "MEL"
]

show_chunk(melanoma_chunks[1])


# ------------------------------------------------------------
# 5. BCC chunk
# ------------------------------------------------------------

print("\n### DERMNET — BCC ###\n")

bcc_chunks = [
    c for c in chunks
    if c["ham10000_class"] == "BCC"
]

show_chunk(bcc_chunks[0])


# ------------------------------------------------------------
# 6. Actinic keratosis chunk
# ------------------------------------------------------------

print("\n### DERMNET — ACTINIC KERATOSIS ###\n")

akiec_chunks = [
    c for c in chunks
    if c["ham10000_class"] == "AKIEC"
]

show_chunk(akiec_chunks[0])


### WHO — FIRST CHUNK ###

CHUNK ID       : who_GENERAL_0000
SOURCE         : WHO
TITLE          : Common Skin Diseases for Management or Referral at Primary Health Care Level
DISEASE        : Common skin diseases
HAM10000 CLASS : None
CHUNK NUMBER   : 0
WORD COUNT     : 800
--------------------------------------------------------------------------------
--- PAGE 1 --- Common Skin Diseases for Management or Referral at Primary Health Care Level WHO guidance with ICD -11 Mortality and Morbidity Statistics (MMS) codes, release 2026 -01 Version 1.0 For external expert review draft --- PAGE 2 --- Disclaimer This document has been prepared to support the recognition, management and referral of common skin diseases at the primary health care (PHC) level. It is intended to assist Ministries of Health, programme managers, educators and primary health care providers in strengthening integrated skin health services. The classification supports clinical decision-making, training, service deliver

# Improve the chunking

In [17]:
INPUT_DIR = "/kaggle/working/clean_text"
OUTPUT_FILE = "/kaggle/working/rag_chunks_v2.json"

# Target size
CHUNK_SIZE = 700
OVERLAP = 120

with open("/kaggle/working/source_metadata.json", "r") as f:
    metadata = json.load(f)

# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def clean_paragraph(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_into_paragraphs(text):
    # Split on blank lines first
    paragraphs = re.split(r"\n\s*\n+", text)

    cleaned = []
    for p in paragraphs:
        p = clean_paragraph(p)
        if p:
            cleaned.append(p)

    return cleaned


def word_count(text):
    return len(text.split())


# ---------------------------------------------------------
# Load metadata by filename
# ---------------------------------------------------------

metadata_by_file = {
    item["filename"]: item
    for item in metadata
}


# ---------------------------------------------------------
# Create section-aware chunks
# ---------------------------------------------------------

all_chunks = []

for filename in sorted(os.listdir(INPUT_DIR)):

    if not filename.endswith(".txt"):
        continue

    filepath = os.path.join(INPUT_DIR, filename)

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    paragraphs = split_into_paragraphs(text)

    source_info = metadata_by_file[filename]

    current = []
    current_words = 0
    chunk_number = 0

    for paragraph in paragraphs:

        p_words = word_count(paragraph)

        # Skip extremely tiny fragments
        if p_words < 5:
            continue

        # If adding paragraph exceeds target,
        # save current chunk first.
        if current and current_words + p_words > CHUNK_SIZE:

            chunk_text = "\n\n".join(current)

            chunk_id = (
                f"{source_info['ham10000_class'] or 'GENERAL'}"
                f"_{chunk_number:04d}"
            )

            all_chunks.append({
                "chunk_id": chunk_id,
                "filename": filename,
                "source": source_info["source"],
                "title": source_info["title"],
                "disease": source_info["disease"],
                "ham10000_class": source_info["ham10000_class"],
                "chunk_number": chunk_number,
                "word_count": word_count(chunk_text),
                "text": chunk_text
            })

            chunk_number += 1

            # -------------------------------------------------
            # Keep overlap using complete paragraphs
            # -------------------------------------------------

            overlap_paragraphs = []
            overlap_words = 0

            for old_p in reversed(current):

                old_words = word_count(old_p)

                if overlap_words + old_words > OVERLAP:
                    break

                overlap_paragraphs.insert(0, old_p)
                overlap_words += old_words

            current = overlap_paragraphs
            current_words = overlap_words

        current.append(paragraph)
        current_words += p_words

    # ---------------------------------------------------------
    # Save final chunk
    # ---------------------------------------------------------

    if current:

        chunk_text = "\n\n".join(current)

        chunk_id = (
            f"{source_info['ham10000_class'] or 'GENERAL'}"
            f"_{chunk_number:04d}"
        )

        all_chunks.append({
            "chunk_id": chunk_id,
            "filename": filename,
            "source": source_info["source"],
            "title": source_info["title"],
            "disease": source_info["disease"],
            "ham10000_class": source_info["ham10000_class"],
            "chunk_number": chunk_number,
            "word_count": word_count(chunk_text),
            "text": chunk_text
        })


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)


# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

print("SECTION-AWARE CHUNKING COMPLETE")
print("=" * 60)

print(f"Total chunks : {len(all_chunks)}")
print(f"Target size  : {CHUNK_SIZE} words")
print(f"Overlap      : {OVERLAP} words")

print("\nChunks by source:")

source_counts = {}

for chunk in all_chunks:
    source = chunk["source"]
    source_counts[source] = source_counts.get(source, 0) + 1

for source, count in source_counts.items():
    print(f"{source:20s}: {count}")

sizes = [c["word_count"] for c in all_chunks]

print("\nChunk word-count statistics:")
print(f"Minimum : {min(sizes)}")
print(f"Maximum : {max(sizes)}")
print(f"Average : {sum(sizes)/len(sizes):.1f}")

print(f"\nSaved to: {OUTPUT_FILE}")

SECTION-AWARE CHUNKING COMPLETE
Total chunks : 46
Target size  : 700 words
Overlap      : 120 words

Chunks by source:
DermNet             : 21
WHO                 : 25

Chunk word-count statistics:
Minimum : 212
Maximum : 758
Average : 605.6

Saved to: /kaggle/working/rag_chunks_v2.json


# Inspect the new chunks

In [18]:
CHUNK_FILE = "/kaggle/working/rag_chunks_v2.json"

with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("=" * 80)
print("RAG CHUNK QUALITY INSPECTION")
print("=" * 80)


def show_chunk(index):
    c = chunks[index]

    print("\n" + "=" * 80)
    print(f"INDEX        : {index}")
    print(f"CHUNK ID     : {c['chunk_id']}")
    print(f"SOURCE       : {c['source']}")
    print(f"DISEASE      : {c['disease']}")
    print(f"HAM10000     : {c['ham10000_class']}")
    print(f"WORD COUNT   : {c['word_count']}")
    print("-" * 80)
    print(c["text"][:3500])


# ---------------------------------------------------------
# 1. First WHO chunk
# ---------------------------------------------------------
print("\n### WHO — FIRST CHUNK")
show_chunk(0)


# ---------------------------------------------------------
# 2. Middle WHO chunk
# ---------------------------------------------------------
who_indices = [
    i for i, c in enumerate(chunks)
    if c["source"] == "WHO"
]

middle_who = who_indices[len(who_indices) // 2]

print("\n### WHO — MIDDLE CHUNK")
show_chunk(middle_who)


# ---------------------------------------------------------
# 3. Last WHO chunk
# ---------------------------------------------------------
print("\n### WHO — FINAL CHUNK")
show_chunk(who_indices[-1])


# ---------------------------------------------------------
# 4. Melanoma chunk
# ---------------------------------------------------------
mel_indices = [
    i for i, c in enumerate(chunks)
    if c["ham10000_class"] == "MEL"
]

print("\n### DERMNET — MELANOMA")
show_chunk(mel_indices[0])


# ---------------------------------------------------------
# 5. BCC chunk
# ---------------------------------------------------------
bcc_indices = [
    i for i, c in enumerate(chunks)
    if c["ham10000_class"] == "BCC"
]

print("\n### DERMNET — BCC")
show_chunk(bcc_indices[0])


# ---------------------------------------------------------
# 6. Actinic keratosis
# ---------------------------------------------------------
ak_indices = [
    i for i, c in enumerate(chunks)
    if c["ham10000_class"] == "AKIEC"
]

print("\n### DERMNET — ACTINIC KERATOSIS")
show_chunk(ak_indices[0])

RAG CHUNK QUALITY INSPECTION

### WHO — FIRST CHUNK

INDEX        : 0
CHUNK ID     : AKIEC_0000
SOURCE       : DermNet
DISEASE      : Actinic keratosis
HAM10000     : AKIEC
WORD COUNT   : 685
--------------------------------------------------------------------------------
Authors: Dr lan Coulson, Dermatologist, United Kingdom; Editor-in-Chief of DermNet (2024). Minor update November 2025. Previous contributors: Dr Amanda Oakley, Dermatologist, Hamilton, New Zealand, (1897); further updated December 2015.

Edited by the DermNet content department.

What is an actinic keratosis?

Actinic keratosis is a precancerous scaly spot found on sun-damaged skin, also known as solar keratosis. It may be considered an early form of cutaneous squamous cell carcinoma (a keratinocyte cancer).

& f oe Te Whatu Ora

Apink base with a hyperkeratotic top on the nasal bridge-a An actinic keratosis on the nose common site for actinic keratoses

A rough scally lesion on the back of the hand - a common site fo

# Install the embedding/vector-search libraries

In [19]:
!pip install -q sentence-transformers faiss-cpu

In [20]:
import torch

print("PyTorch:", torch.__version__)

try:
    import torchvision
    print("Torchvision:", torchvision.__version__)
except Exception as e:
    print("Torchvision error:", e)

PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128


In [21]:
import sentence_transformers
import faiss

print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)

print("\nEmbedding/vector libraries ready.")

Sentence Transformers: 5.4.1
FAISS: 1.15.1

Embedding/vector libraries ready.


# Load the embedding model

In [22]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model...")
embedding_model = SentenceTransformer(MODEL_NAME)

print("\nEmbedding model loaded successfully.")
print("Model:", MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully.
Model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_58/1708439071.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


# Generate embeddings for all 46 chunks

In [23]:
import numpy as np

# Load our section-aware RAG chunks
CHUNKS_PATH = "/kaggle/working/rag_chunks_v2.json"

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Loaded chunks:", len(chunks))

# Extract chunk text
texts = [chunk["text"] for chunk in chunks]

print("Generating embeddings...")

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("\nEMBEDDINGS GENERATED")
print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("Min value:", embeddings.min())
print("Max value:", embeddings.max())

Loaded chunks: 46
Generating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


EMBEDDINGS GENERATED
Shape: (46, 384)
Data type: float32
Min value: -0.20128012
Max value: 0.21577203


# Build the FAISS vector index

In [24]:
import faiss

# Make sure embeddings are float32
embeddings = np.asarray(embeddings, dtype="float32")

# Create FAISS index using inner product.
# Because embeddings were normalized, inner product = cosine similarity.
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS INDEX CREATED")
print("Embedding dimension:", embedding_dim)
print("Vectors in index:", index.ntotal)
print("Index type:", type(index).__name__)

# Save index
INDEX_PATH = "/kaggle/working/skinova_faiss.index"
faiss.write_index(index, INDEX_PATH)

print("\nSaved to:", INDEX_PATH)
print("File exists:", os.path.exists(INDEX_PATH))
print("File size (MB):", round(os.path.getsize(INDEX_PATH) / (1024**2), 3))

FAISS INDEX CREATED
Embedding dimension: 384
Vectors in index: 46
Index type: IndexFlatIP

Saved to: /kaggle/working/skinova_faiss.index
File exists: True
File size (MB): 0.067


# Test the RAG retrieval

In [25]:


def search_rag(query, top_k=5):
    """
    Search the FAISS index and return the most relevant knowledge chunks.
    """
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[int(idx)]

        results.append({
            "score": float(score),
            "source": chunk["source"],
            "filename": chunk["filename"],
            "ham10000_class": chunk.get("ham10000_class"),
            "text": chunk["text"]
        })

    return results


# Test query
query = "What are the clinical features and risk factors of melanoma?"

results = search_rag(query, top_k=5)

print("=" * 80)
print("QUERY:", query)
print("=" * 80)

for i, result in enumerate(results, 1):
    print(f"\nRESULT {i}")
    print("-" * 80)
    print("Similarity score :", round(result["score"], 4))
    print("Source           :", result["source"])
    print("HAM10000 class   :", result["ham10000_class"])
    print("Filename         :", result["filename"])
    print("\nText:")
    print(result["text"][:1200])

QUERY: What are the clinical features and risk factors of melanoma?

RESULT 1
--------------------------------------------------------------------------------
Similarity score : 0.6852
Source           : DermNet
HAM10000 class   : MEL
Filename         : dermnet_melanoma_clean.txt

Text:
--- PAGE 13 --- Most tests are not worthwhile for patients with stage | or 2 melanoma unless there are signs or symptoms of disease recurrence or metastasis. No tests are necessary for healthy patients who have remained well for five years or longer after the removal of their melanoma.

How do you prevent melanoma?

Preventative measures involve addressing risk factors such as exposure to UV radiation, eg, wearing protective clothing, using sunscreen (SPF 50), and avoiding tanning beds. For more information, see skin

What is the outcome of melanoma?

Melanoma in situ is cured by excision because it has no potential to spread around the body.

The risk of spread and ultimate death from invasive melanoma

# Test retrieval across all 7 classes

In [26]:
test_queries = {
    "MEL": "What are the clinical features and risk factors of melanoma?",
    "NV": "What are the clinical features and characteristics of melanocytic naevi?",
    "BCC": "What are the clinical features and risk factors of basal cell carcinoma?",
    "AKIEC": "What are the clinical features and risk factors of actinic keratosis?",
    "BKL": "What are the clinical features and characteristics of seborrhoeic keratosis?",
    "DF": "What are the clinical features and characteristics of dermatofibroma?",
    "VASC": "What are the clinical features and characteristics of vascular lesions?"
}

for expected_class, query in test_queries.items():

    results = search_rag(query, top_k=3)

    print("\n" + "=" * 90)
    print(f"EXPECTED CLASS: {expected_class}")
    print(f"QUERY: {query}")
    print("=" * 90)

    for i, result in enumerate(results, 1):
        print(
            f"{i}. "
            f"Score={result['score']:.4f} | "
            f"Class={result['ham10000_class']} | "
            f"Source={result['source']}"
        )


EXPECTED CLASS: MEL
QUERY: What are the clinical features and risk factors of melanoma?
1. Score=0.6852 | Class=MEL | Source=DermNet
2. Score=0.6594 | Class=MEL | Source=DermNet
3. Score=0.6387 | Class=MEL | Source=DermNet

EXPECTED CLASS: NV
QUERY: What are the clinical features and characteristics of melanocytic naevi?
1. Score=0.6804 | Class=NV | Source=DermNet
2. Score=0.6559 | Class=MEL | Source=DermNet
3. Score=0.6478 | Class=NV | Source=DermNet

EXPECTED CLASS: BCC
QUERY: What are the clinical features and risk factors of basal cell carcinoma?
1. Score=0.6925 | Class=BCC | Source=DermNet
2. Score=0.6634 | Class=BCC | Source=DermNet
3. Score=0.5333 | Class=None | Source=WHO

EXPECTED CLASS: AKIEC
QUERY: What are the clinical features and risk factors of actinic keratosis?
1. Score=0.6898 | Class=AKIEC | Source=DermNet
2. Score=0.6856 | Class=AKIEC | Source=DermNet
3. Score=0.5293 | Class=BKL | Source=DermNet

EXPECTED CLASS: BKL
QUERY: What are the clinical features and characte

# Save the complete RAG knowledge base

In [27]:

RAG_DIR = "/kaggle/working/skinova_rag"
os.makedirs(RAG_DIR, exist_ok=True)

# --------------------------------------------------
# 1. Save chunks
# --------------------------------------------------

chunks_path = os.path.join(RAG_DIR, "chunks.json")

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)


# --------------------------------------------------
# 2. Save embeddings
# --------------------------------------------------

embeddings_path = os.path.join(RAG_DIR, "embeddings.npy")

np.save(
    embeddings_path,
    np.asarray(embeddings, dtype="float32")
)


# --------------------------------------------------
# 3. Save FAISS index
# --------------------------------------------------

index_path = os.path.join(RAG_DIR, "faiss.index")

faiss.write_index(index, index_path)


# --------------------------------------------------
# 4. Save source metadata
# --------------------------------------------------

metadata_source = "/kaggle/working/source_metadata.json"
metadata_path = os.path.join(RAG_DIR, "source_metadata.json")

with open(metadata_source, "r", encoding="utf-8") as f:
    source_metadata = json.load(f)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(source_metadata, f, ensure_ascii=False, indent=2)


# --------------------------------------------------
# 5. Save model information
# --------------------------------------------------

model_info = {
    "embedding_model": MODEL_NAME,
    "embedding_dimension": int(embeddings.shape[1]),
    "normalization": "L2 normalized",
    "similarity": "cosine similarity via inner product",
    "num_chunks": len(chunks),
    "num_vectors": int(index.ntotal)
}

model_info_path = os.path.join(RAG_DIR, "model_info.json")

with open(model_info_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)


# --------------------------------------------------
# Verify
# --------------------------------------------------

print("=" * 70)
print("SKINOVA RAG KNOWLEDGE BASE SAVED")
print("=" * 70)

for filename in sorted(os.listdir(RAG_DIR)):
    filepath = os.path.join(RAG_DIR, filename)
    size_mb = os.path.getsize(filepath) / (1024 ** 2)
    print(f"{filename:<25} {size_mb:.3f} MB")

print("\nChunks :", len(chunks))
print("Vectors:", index.ntotal)
print("Dimension:", embeddings.shape[1])
print("\nRAG directory:", RAG_DIR)

SKINOVA RAG KNOWLEDGE BASE SAVED
chunks.json               0.201 MB
embeddings.npy            0.068 MB
faiss.index               0.067 MB
model_info.json           0.000 MB
source_metadata.json      0.002 MB

Chunks : 46
Vectors: 46
Dimension: 384

RAG directory: /kaggle/working/skinova_rag


# Create the retrieve() function

In [28]:
def retrieve(query, top_k=5, min_score=0.0):
    """
    Retrieve the most relevant knowledge chunks for a query.

    Parameters
    ----------
    query : str
        User/clinical question.
    top_k : int
        Number of chunks to retrieve.
    min_score : float
        Optional minimum cosine similarity score.

    Returns
    -------
    list
        Retrieved chunks with scores and source information.
    """

    # Encode query
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        # Ignore invalid FAISS indices
        if idx < 0:
            continue

        score = float(score)

        # Optional score filtering
        if score < min_score:
            continue

        chunk = chunks[int(idx)]

        results.append({
            "rank": len(results) + 1,
            "score": round(score, 4),
            "source": chunk["source"],
            "filename": chunk["filename"],
            "ham10000_class": chunk.get("ham10000_class"),
            "text": chunk["text"]
        })

    return results


print("retrieve() function created successfully.")

retrieve() function created successfully.


# Create the RAG context formatter

In [29]:
def format_rag_context(results):
    """
    Convert retrieved RAG results into a structured context
    that can be passed to an LLM.
    """

    if not results:
        return "No relevant knowledge was retrieved."

    context_parts = []

    for result in results:
        part = f"""
SOURCE {result['rank']}
Source: {result['source']}
Document: {result['filename']}
HAM10000 class: {result['ham10000_class']}
Similarity: {result['score']}

Knowledge:
{result['text']}
"""
        context_parts.append(part.strip())

    return "\n\n" + "\n\n".join(context_parts)


# Test the formatter
query = "What are the clinical features of basal cell carcinoma?"

results = retrieve(query, top_k=3)

context = format_rag_context(results)

print("=" * 80)
print("RAG CONTEXT")
print("=" * 80)
print(context[:6000])

RAG CONTEXT


SOURCE 1
Source: DermNet
Document: dermnet_bcc_clean.txt
HAM10000 class: BCC
Similarity: 0.6773

Knowledge:
What are the complications of basal cell carcinoma?

Recurrence of BCC after initial treatment is not uncommon. Characteristics of recurrent BCC often include:

Incomplete excision or narrow margins at primary excision Morphoeic, micronodular, and infiltrative subtypes Location on head and neck.

After PDT After superficial surgery

--- PAGE 6 --- Te Whatu Ora

Advanced BCCs are large, often neglected tumours.

They may be several centimetres in diameter They may be deeply infiltrating into tissues below the skin They are difficult or impossible to treat surgically

Primary tumour is often large, neglected or recurrent, located on head and neck, with aggressive subtype

May have had multiple prior treatments

May arise in-site exposed to ionising radiation Can be fatal

How is basal cell carcinoma diagnosed?

BCC is diagnosed clinically by the presence of a slowly e

# Class-aware retrieval

In [30]:
def retrieve_class_aware(
    query,
    predicted_class=None,
    top_k=3,
    candidate_k=None,
    class_bonus=0.10,
    min_class_chunks=2,
):
    """
    Retrieve knowledge using semantic similarity plus the predicted HAM10000 class.

    - Only ~46 chunks exist, so we score ALL of them (candidate_k=None) instead of
      a top-15 pool that could miss the predicted class entirely.
    - Chunks tagged with the predicted class get a ranking bonus.
    - At least `min_class_chunks` chunks of the predicted class are guaranteed
      to be in the result (if that many exist). WHO/general chunks can still appear.
    """
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    candidate_k = min(candidate_k or index.ntotal, index.ntotal)
    scores, indices = index.search(query_embedding, candidate_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        chunk = chunks[int(idx)]
        is_match = predicted_class is not None and chunk.get("ham10000_class") == predicted_class
        candidates.append({
            "original_score": float(score),
            "ranking_score": float(score) + (class_bonus if is_match else 0.0),
            "chunk_id": chunk.get("chunk_id"),
            "source": chunk["source"],
            "filename": chunk["filename"],
            "ham10000_class": chunk.get("ham10000_class"),
            "text": chunk["text"],
        })

    candidates.sort(key=lambda x: x["ranking_score"], reverse=True)
    selected = candidates[:top_k]

    if predicted_class is not None:
        need_total = min(min_class_chunks, top_k)
        have = sum(c["ham10000_class"] == predicted_class for c in selected)
        if have < need_total:
            extras = [
                c for c in candidates
                if c["ham10000_class"] == predicted_class and c not in selected
            ]
            for extra in extras[: need_total - have]:
                # drop the weakest non-class chunk to make room
                for j in range(len(selected) - 1, -1, -1):
                    if selected[j]["ham10000_class"] != predicted_class:
                        selected.pop(j)
                        break
                selected.append(extra)
            selected.sort(key=lambda x: x["ranking_score"], reverse=True)

    results = []
    for rank, c in enumerate(selected, 1):
        results.append({
            "rank": rank,
            "score": round(c["original_score"], 4),
            "chunk_id": c["chunk_id"],
            "source": c["source"],
            "filename": c["filename"],
            "ham10000_class": c["ham10000_class"],
            "text": c["text"],
        })

    return results


print("Class-aware retrieval function created successfully.")


Class-aware retrieval function created successfully.


In [31]:
### Test

query = "What are the clinical features and risk factors of basal cell carcinoma?"

predicted_class = "BCC"

results = retrieve_class_aware(
    query=query,
    predicted_class=predicted_class,
    top_k=5
)

print("=" * 80)
print("QUERY:", query)
print("PREDICTED CLASS:", predicted_class)
print("=" * 80)

for result in results:
    print(
        f"\nRank {result['rank']} | "
        f"Score: {result['score']} | "
        f"Class: {result['ham10000_class']} | "
        f"Source: {result['source']}"
    )

QUERY: What are the clinical features and risk factors of basal cell carcinoma?
PREDICTED CLASS: BCC

Rank 1 | Score: 0.6925 | Class: BCC | Source: DermNet

Rank 2 | Score: 0.6634 | Class: BCC | Source: DermNet

Rank 3 | Score: 0.5333 | Class: None | Source: WHO

Rank 4 | Score: 0.4878 | Class: MEL | Source: DermNet

Rank 5 | Score: 0.4708 | Class: MEL | Source: DermNet


# Build the final evidence package

In [32]:
CLASS_LABELS = {
    "MEL": "melanoma",
    "NV": "melanocytic naevus (mole)",
    "BCC": "basal cell carcinoma",
    "AKIEC": "actinic keratosis",
    "BKL": "seborrhoeic keratosis (benign keratosis)",
    "DF": "dermatofibroma",
    "VASC": "vascular lesion",
}


def augment_query(query, predicted_class):
    label = CLASS_LABELS.get(predicted_class)
    if not label:
        return query
    return f"{label}: clinical features, risk factors, management and when to see a doctor. {query}"


def build_evidence_package(
    query,
    predicted_class,
    prediction_confidence=None,
    top_k=5
):
    """
    Build a structured evidence package for the Skinova LLM.

    Parameters
    ----------
    query : str
        User's question or explanation request.

    predicted_class : str
        EfficientNetB0 predicted HAM10000 class.

    prediction_confidence : float, optional
        Model confidence, e.g. 0.87.

    top_k : int
        Number of RAG chunks to retrieve.
    """

    # Retrieve class-aware medical knowledge
    # A generic question ("what is this?") barely matches any medical text, so
    # add the predicted disease name to the *retrieval* query only.
    retrieval_query = augment_query(query, predicted_class)

    retrieved = retrieve_class_aware(
        query=retrieval_query,
        predicted_class=predicted_class,
        top_k=top_k
    )

    # Format retrieved evidence
    context = format_rag_context(retrieved)

    # Build structured package
    package = {
        "query": query,
        "prediction": {
            "ham10000_class": predicted_class,
            "confidence": prediction_confidence
        },
        "retrieval": {
            "num_results": len(retrieved),
            "results": retrieved
        },
        "llm_context": context
    }

    return package


# --------------------------------------------------
# Test the evidence package
# --------------------------------------------------

evidence = build_evidence_package(
    query="What are the clinical features and risk factors of basal cell carcinoma?",
    predicted_class="BCC",
    prediction_confidence=0.91,
    top_k=5
)

print("=" * 80)
print("SKINOVA EVIDENCE PACKAGE")
print("=" * 80)

print("\nPrediction:")
print(evidence["prediction"])

print("\nRetrieved sources:")

for result in evidence["retrieval"]["results"]:
    print(
        f"  {result['rank']}. "
        f"{result['source']} | "
        f"{result['ham10000_class']} | "
        f"score={result['score']}"
    )

print("\nLLM context preview:")
print(evidence["llm_context"][:5000])

SKINOVA EVIDENCE PACKAGE

Prediction:
{'ham10000_class': 'BCC', 'confidence': 0.91}

Retrieved sources:
  1. DermNet | BCC | score=0.6897
  2. DermNet | BCC | score=0.6161
  3. WHO | None | score=0.459
  4. DermNet | MEL | score=0.4455
  5. WHO | None | score=0.4064

LLM context preview:


SOURCE 1
Source: DermNet
Document: dermnet_bcc_clean.txt
HAM10000 class: BCC
Similarity: 0.6897

Knowledge:
--- PAGE 1 --- == DermNet°®

Authors: Honorary Associate Profes Anthony Martin Fuentes, General Practitioner and Cosmetic

Update peer reviewed by: Dr Andje bourne Hospital, Australia, (2028)

Reviewing dermatologist: Dr lan Ce Edited by the DermNet content department.

What is basal cell carcinoma?

Basal cell carcinoma (BCC) is a common, locally invasive, keratinocyte cancer (also known as nonmelanoma cancer). It is the most common form of skin cancer. BCC is also known as rodent ulcer and basalioma. Patients with BCC often develop multiple primary tumours over time.

BCC is also known as rod

# Create the Skinova LLM prompt

In [33]:
def build_skinova_prompt(evidence):
    """
    Build the final grounded prompt for the Skinova LLM.
    """

    prediction = evidence["prediction"]
    retrieved = evidence["retrieval"]["results"]

    predicted_class = prediction["ham10000_class"]
    confidence = prediction["confidence"]

    # Convert confidence to readable form
    if confidence is not None:
        confidence_text = f"{confidence:.2%}"
    else:
        confidence_text = "Not provided"

    # Build source evidence
    evidence_blocks = []

    for result in retrieved:
        evidence_blocks.append(
            f"""
SOURCE {result['rank']}
Source: {result['source']}
Document: {result['filename']}
HAM10000 class tag: {result['ham10000_class']}
Similarity score: {result['score']}

{result['text']}
""".strip()
        )

    evidence_text = "\n\n".join(evidence_blocks)

    prompt = f"""
You are Skinova, an AI-assisted dermatology support system.

Your task is to provide a clear, cautious, knowledge-grounded explanation
based on an image classification result and retrieved medical references.

IMAGE CLASSIFICATION RESULT
Predicted HAM10000 class: {predicted_class}
Classifier confidence: {confidence_text}

IMPORTANT:
- The image classifier prediction is an AI prediction, not a confirmed diagnosis.
- Do not state that the patient definitely has the predicted condition.
- Do not invent clinical findings that are not present in the supplied evidence.
- Use the retrieved medical sources to explain the predicted class.
- Prioritize retrieved evidence whose HAM10000 class tag matches the predicted class.
- WHO/general evidence may be used when relevant.
- Do not use evidence tagged with a different disease class as if it were
  evidence about the predicted class.
- If the retrieved sources do not support a claim, say that the available
  sources do not provide enough information.
- Do not fabricate citations, statistics, treatments, or recommendations.
- Clearly distinguish the classifier prediction from information contained
  in the medical references.
- For medical decisions, emphasize that appropriate clinical assessment
  and professional evaluation are required.

RETRIEVED MEDICAL EVIDENCE
==========================

{evidence_text}

RESPONSE REQUIREMENTS
=====================

Answer the user's question using the evidence above.

Structure the response as appropriate, using sections such as:

1. AI prediction
2. What the retrieved medical sources say
3. Relevant clinical features
4. Relevant risk factors or considerations
5. Important limitations

Keep the explanation understandable to a non-specialist while retaining
medically important terminology. Keep the whole answer under about 350 words.

Do not present the AI prediction as a definitive diagnosis.
""".strip()

    return prompt


# --------------------------------------------------
# Test prompt generation
# --------------------------------------------------

skinova_prompt = build_skinova_prompt(evidence)

print("=" * 80)
print("SKINOVA LLM PROMPT")
print("=" * 80)
print(skinova_prompt[:10000])

SKINOVA LLM PROMPT
You are Skinova, an AI-assisted dermatology support system.

Your task is to provide a clear, cautious, knowledge-grounded explanation
based on an image classification result and retrieved medical references.

IMAGE CLASSIFICATION RESULT
Predicted HAM10000 class: BCC
Classifier confidence: 91.00%

IMPORTANT:
- The image classifier prediction is an AI prediction, not a confirmed diagnosis.
- Do not state that the patient definitely has the predicted condition.
- Do not invent clinical findings that are not present in the supplied evidence.
- Use the retrieved medical sources to explain the predicted class.
- Prioritize retrieved evidence whose HAM10000 class tag matches the predicted class.
- WHO/general evidence may be used when relevant.
- Do not use evidence tagged with a different disease class as if it were
  evidence about the predicted class.
- If the retrieved sources do not support a claim, say that the available
  sources do not provide enough information.
-

# Choose and connect the LLM

In [34]:
import torch

In [35]:
print("=" * 70)
print("SKINOVA LLM ENVIRONMENT")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB"
    )
else:
    print("GPU: Not available")

print("\nPython:", os.sys.version.split()[0])

SKINOVA LLM ENVIRONMENT
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

Python: 3.12.13


In [36]:
print("=" * 70)
print("SKINOVA LLM ENVIRONMENT")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
    print(
        f"GPU {i} memory:",
        round(torch.cuda.get_device_properties(i).total_memory / (1024**3), 2),
        "GB"
    )

print("\nPython:", os.sys.version.split()[0])

SKINOVA LLM ENVIRONMENT
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 0 memory: 14.56 GB
GPU 1: Tesla T4
GPU 1 memory: 14.56 GB

Python: 3.12.13


# Install the LLM packages

In [37]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.0 MB/s eta 0:00:00:00:01


# Verify the LLM packages

In [38]:
import transformers
import accelerate
import bitsandbytes
import torch

print("=" * 70)
print("SKINOVA LLM PACKAGES")
print("=" * 70)

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

SKINOVA LLM PACKAGES
Transformers: 5.0.0
Accelerate: 1.13.0
BitsAndBytes: 0.50.2
CUDA available: True
GPU count: 2


# Download and load Qwen2.5-7B-Instruct

In [39]:
!pip install -q --no-cache-dir --force-reinstall \
    "transformers==5.0.0" \
    "accelerate==1.13.0" \
    "bitsandbytes==0.50.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 229.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 165.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 121.4 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 383.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 295.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 364.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 292.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 211.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 357.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 322.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.6/804.6 kB 213.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 293.7 MB/s eta 0:00:00
   ━━━

In [40]:
import huggingface_hub
print("=" * 70)
print("SKINOVA LLM ENVIRONMENT — FINAL CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

SKINOVA LLM ENVIRONMENT — FINAL CHECK
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
Hugging Face Hub: 1.11.0
Accelerate: 1.13.0
BitsAndBytes: 0.50.2

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [41]:
import sys

sys.path.insert(0, "/kaggle/working/skinova_llm_env")

In [42]:
import os

llm_env = "/kaggle/working/skinova_llm_env"

print("=" * 70)
print("SKINOVA ISOLATED LLM ENVIRONMENT")
print("=" * 70)

print("Environment exists:",
      os.path.exists(llm_env))

print("Transformers package:",
      os.path.exists(os.path.join(llm_env, "transformers")))

print("Hugging Face Hub package:",
      os.path.exists(os.path.join(llm_env, "huggingface_hub")))

print("Tokenizers package:",
      os.path.exists(os.path.join(llm_env, "tokenizers")))

SKINOVA ISOLATED LLM ENVIRONMENT
Environment exists: False
Transformers package: False
Hugging Face Hub package: False
Tokenizers package: False


# Activate the isolated LLM packages

In [43]:
import sys

llm_env = "/kaggle/working/skinova_llm_env"
if llm_env not in sys.path:
    sys.path.insert(0, llm_env)   # put the isolated packages FIRST on the path

# First-ever import of these — no reload needed
import transformers
import huggingface_hub
import tokenizers

print("Transformers loaded from:", transformers.__file__)
print("Transformers version:", transformers.__version__)


# Run this in a FRESH kernel (Kaggle: Run > Restart & Run All, or restart session
# and re-run cells 0-67 first so pdf_paths/chunks/index/embedding_model etc. exist
# again, but do NOT import transformers anywhere until this cell runs).

import sys

llm_env = "/kaggle/working/skinova_llm_env"
if llm_env not in sys.path:
    sys.path.insert(0, llm_env)   # put the isolated packages FIRST on the path

# First-ever import of these — no reload needed
import transformers
import huggingface_hub
import tokenizers

print("Transformers loaded from:", transformers.__file__)
print("Transformers version:", transformers.__version__)

Transformers loaded from: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Transformers version: 5.0.0
Transformers loaded from: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Transformers version: 5.0.0


In [44]:
#%pip install --no-cache-dir --force-reinstall "transformers==4.56.2" "tokenizers>=0.22.0,<0.23.0"

In [45]:
import transformers
import tokenizers

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)

from transformers import AutoTokenizer

print("AutoTokenizer imported successfully!")

Transformers: 5.0.0
Tokenizers: 0.22.2
AutoTokenizer imported successfully!


## Load Qwen2.5-7B-Instruct in 4-bit

In [46]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit quantization so the 7B model fits on a single Kaggle T4/P100 (~15-16GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)

print("Loading model in 4-bit (this can take a few minutes on first run)...")
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

llm_model.eval()
print("Qwen2.5-7B-Instruct loaded.")
print("Memory footprint (GB):", round(llm_model.get_memory_footprint() / (1024**3), 2))



Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B-Instruct loaded.
Memory footprint (GB): 5.07


##  Generation function

In [47]:
SYSTEM_PROMPT = (
    "You are Skinova, a cautious dermatology information assistant. "
    "You never give a definitive diagnosis and you only use the evidence you are given."
)

# Qwen2.5 supports a long context; this is just a safety net so a prompt is
# never silently cut (the old max_length=2048 chopped off the instructions).
MAX_PROMPT_TOKENS = 7000


def generate_response(
    prompt,
    max_new_tokens=700,
    do_sample=False,      # greedy = reproducible, better for medical text
    temperature=0.3,
    top_p=0.9,
):
    # Instruct models must be prompted through their chat template.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = llm_tokenizer(text, return_tensors="pt")   # NO truncation
    n_prompt = inputs["input_ids"].shape[1]

    if n_prompt > MAX_PROMPT_TOKENS:
        raise ValueError(
            f"Prompt is {n_prompt} tokens (limit {MAX_PROMPT_TOKENS}). "
            "Lower top_k or use smaller chunks instead of truncating."
        )

    inputs = {k: v.to(llm_model.device) for k, v in inputs.items()}

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=llm_tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kwargs.update(temperature=temperature, top_p=top_p)
    else:
        gen_kwargs.update(temperature=None, top_p=None, top_k=None)

    with torch.no_grad():
        output_ids = llm_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            **gen_kwargs,
        )

    generated_ids = output_ids[0][n_prompt:]
    return llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


## Test the LLM alone against a RAG prompt

In [48]:
test_evidence = build_evidence_package(
    query="What does this condition look like and what should I do next?",
    predicted_class="BCC",
    prediction_confidence=0.87,
    top_k=3,
)

test_prompt = build_skinova_prompt(test_evidence)

print("Prompt tokens:", len(llm_tokenizer(test_prompt)["input_ids"]))
test_answer = generate_response(test_prompt)

print("=" * 80)
print("GENERATED ANSWER")
print("=" * 80)
print(test_answer)

Prompt tokens: 3505
GENERATED ANSWER
### AI Prediction
The image classifier predicts the condition to be Basal Cell Carcinoma (BCC) with a high confidence level of 87.00%. However, this prediction should not be considered a definitive diagnosis without further clinical evaluation.

### What the Retrieved Medical Sources Say
The medical literature supports the AI prediction. Basal cell carcinoma (BCC) is the most common form of skin cancer, characterized by its slow growth and local invasiveness. It is commonly found in sun-exposed areas such as the face, ears, scalp, and forearms.

### Relevant Clinical Features
Clinical features of BCC include:
- **Appearance**: Slowly growing, shiny or pearly nodules with a smooth surface, sometimes with a central depression or ulceration. The edges may appear rolled.
- **Location**: Commonly found on the face, especially around the nose, ears, and eyelids, but can occur on any sun-exposed area.
- **Size**: Ranges from a few millimeters to several ce

# Wire the CNN into the pipeline: image -> predicted_class

In [49]:
import os

for dirname, _, filenames in os.walk("/kaggle/working"):
    for filename in filenames:
        if filename.endswith((".keras", ".h5", ".pb")):
            print(os.path.join(dirname, filename))

In [50]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith((".keras", ".h5", ".zip")):
            print(os.path.join(dirname, filename))

/kaggle/input/models/eshamanohar/skinova-effiecientb0-bestmodel/keras/default/1/skinova_efficientnetb0_best.keras


In [51]:
# TensorFlow must NOT use the GPU in this notebook.
# PyTorch (Qwen) already initialised CUDA in this process, so setting
# CUDA_VISIBLE_DEVICES here is too late and has no effect (TF still listed 2 GPUs).
# Two different cuDNN builds (torch's and TF's) in one process is what produced
# "cudnnFinalize failed / unknown cudnn status: 1002".
import tensorflow as tf

try:
    tf.config.set_visible_devices([], "GPU")   # works only before TF creates any op
    print("TensorFlow GPUs hidden.")
except RuntimeError as e:
    # TF already initialised its GPUs in this kernel (e.g. after a failed predict).
    # Fine: the model below is built and run under tf.device("/CPU:0") explicitly.
    print("Could not hide GPUs (already initialised) - using explicit CPU placement:", e)

print("TensorFlow:", tf.__version__)
print("TF visible GPUs:", tf.config.get_visible_devices("GPU"))


TensorFlow GPUs hidden.
TensorFlow: 2.20.0
TF visible GPUs: []


In [52]:
import os
from tensorflow.keras.models import load_model

model_path = "/kaggle/input/models/eshamanohar/skinova-effiecientb0-bestmodel/keras/default/1/skinova_efficientnetb0_best.keras"

print("Model exists:", os.path.isfile(model_path))

# Inference only: build the weights on CPU and skip the optimizer state.
with tf.device("/CPU:0"):
    skin_cnn = load_model(model_path, compile=False)

print("SKINOVA EfficientNetB0 loaded successfully (CPU).")


Model exists: True
SKINOVA EfficientNetB0 loaded successfully (CPU).


In [53]:
skin_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,384,426 (16.73 MB)

 Trainable params: 1,815,655 (6.93 MB)

 Non-trainable params: 2,568,771 (9.80 MB)

In [54]:
print("Input shape:", skin_cnn.input_shape)
print("Output shape:", skin_cnn.output_shape)

Input shape: (None, 240, 240, 3)
Output shape: (None, 7)


In [55]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array

# skin_cnn was already loaded (on CPU) two cells above - no second load_model here.

# Must match the class order used when training (CLASSES in the CNN notebook).
# The check cell below prints a confusion matrix so a wrong order is visible.
HAM10000_CLASSES = ["AKIEC", "BCC", "BKL", "DF", "MEL", "NV", "VASC"]
IMG_SIZE = (240, 240)

# Does the model expect pixels in 0-1 or 0-255?  Keras' EfficientNet normally
# rescales internally, so dividing by 255 AGAIN can wreck predictions.
# The next cell tests both options and sets this flag for you.
DIVIDE_BY_255 = True


def load_raw_image(image_path):
    """Return a float32 array (240, 240, 3) with values 0-255."""
    img = load_img(image_path, target_size=IMG_SIZE, color_mode="rgb")
    return img_to_array(img).astype("float32")


def preprocess_image(image_path):
    arr = load_raw_image(image_path)
    if DIVIDE_BY_255:
        arr = arr / 255.0
    return np.expand_dims(arr, axis=0)


def predict_probs(image_path):
    x = preprocess_image(image_path)
    # Direct eager call on CPU: no cached GPU predict-function, no cuDNN.
    with tf.device("/CPU:0"):
        probs = skin_cnn(x, training=False).numpy()[0]
    return probs


def predict_topk(image_path, k=3):
    probs = predict_probs(image_path)
    order = np.argsort(probs)[::-1][:k]
    return [(HAM10000_CLASSES[i], float(probs[i])) for i in order]


def predict_skin_condition(image_path):
    """Return (predicted_class, confidence) for one image."""
    return predict_topk(image_path, k=1)[0]


## Sanity check: preprocessing and class order
Tests `/255` vs raw 0-255 input on labelled HAM10000 images (20 per class) and prints a confusion matrix. Rows = true class, columns = predicted class.

In [56]:
import glob
import pandas as pd

DATA_ROOT = "/kaggle/input/datasets/surajghuwalewala/ham1000-segmentation-and-classification"

base = skin_cnn.layers[0]
print("Base model:", base.name, "| first layers:", [l.name for l in base.layers[:5]])
print("Built-in Rescaling layer:", any(isinstance(l, tf.keras.layers.Rescaling) for l in base.layers[:5]))

# ---- find labels -------------------------------------------------------------
gt = None
for p in glob.glob(f"{DATA_ROOT}/**/*.csv", recursive=True):
    df = pd.read_csv(p)
    cols = {c.upper(): c for c in df.columns}
    if set(HAM10000_CLASSES).issubset(cols):                    # one-hot GroundTruth.csv
        id_col = next(c for c in df.columns if c.lower() in ("image", "image_id"))
        labels = df[[cols[c] for c in HAM10000_CLASSES]].idxmax(axis=1).str.upper()
        gt = pd.DataFrame({"id": df[id_col], "label": labels})
    elif "dx" in df.columns and "image_id" in df.columns:        # HAM10000_metadata.csv
        gt = pd.DataFrame({"id": df["image_id"], "label": df["dx"].str.upper()})
    if gt is not None:
        print("Labels from:", p)
        break

if gt is None:
    print("No label CSV found - skipping accuracy check. Compare the two options by eye instead.")
else:
    paths = {os.path.splitext(os.path.basename(p))[0]: p
             for p in glob.glob(f"{DATA_ROOT}/**/*.jpg", recursive=True)}
    gt["id"] = gt["id"].astype(str).str.replace(r"\.jpe?g$", "", regex=True)
    gt = gt[gt["id"].isin(paths)]
    if gt.empty:
        raise RuntimeError("Label ids do not match any image file names - check DATA_ROOT.")
    sample = pd.concat([g.sample(min(len(g), 20), random_state=0) for _, g in gt.groupby("label")])

    raw = np.stack([load_raw_image(paths[i]) for i in sample["id"]])
    true = sample["label"].values

    def run(x):
        out = []
        with tf.device("/CPU:0"):
            for s in range(0, len(x), 8):
                out.append(skin_cnn(x[s:s + 8], training=False).numpy())
        return np.concatenate(out)

    results = {}
    for name, x in [("divide by 255", raw / 255.0), ("raw 0-255", raw)]:
        pred = np.array(HAM10000_CLASSES)[run(x).argmax(1)]
        recalls = [(pred[true == c] == c).mean() for c in HAM10000_CLASSES if (true == c).any()]
        results[name] = (float(np.mean(recalls)), pred)
        print(f"{name:14s}  balanced accuracy = {np.mean(recalls):.3f}")

    best = max(results, key=lambda k: results[k][0])
    DIVIDE_BY_255 = (best == "divide by 255")
    print(f"\n-> using '{best}'  (DIVIDE_BY_255 = {DIVIDE_BY_255})")
    print("\nConfusion matrix (rows = true, cols = predicted):")
    print(pd.crosstab(pd.Series(true, name="true"), pd.Series(results[best][1], name="pred")))
    print("\nNote: these images may include training data, so treat the numbers as a wiring check, "
          "not as a performance estimate. If one column dominates or the diagonal is off, "
          "HAM10000_CLASSES is in the wrong order.")


Base model: efficientnetb0 | first layers: ['input_layer', 'rescaling', 'normalization', 'rescaling_1', 'stem_conv_pad']
Built-in Rescaling layer: True
Labels from: /kaggle/input/datasets/surajghuwalewala/ham1000-segmentation-and-classification/GroundTruth.csv
divide by 255   balanced accuracy = 0.143
raw 0-255       balanced accuracy = 0.936

-> using 'raw 0-255'  (DIVIDE_BY_255 = False)

Confusion matrix (rows = true, cols = predicted):
pred   AKIEC  BCC  BKL  DF  MEL  NV  VASC
true                                     
AKIEC     18    1    1   0    0   0     0
BCC        1   19    0   0    0   0     0
BKL        0    0   18   0    1   1     0
DF         0    0    0  19    0   1     0
MEL        0    0    0   0   17   3     0
NV         0    0    0   0    0  20     0
VASC       0    0    0   0    0   0    20

Note: these images may include training data, so treat the numbers as a wiring check, not as a performance estimate. If one column dominates or the diagonal is off, HAM10000_CLAS

## Full end-to-end pipeline: image + question -> answer

In [57]:
DISCLAIMER = (
    "This is an educational AI tool, not a medical device, and the result is an "
    "image-based prediction, not a diagnosis. Please have any new, changing, "
    "bleeding or worrying skin lesion examined by a doctor or dermatologist."
)
LOW_CONFIDENCE = 0.60
HIGH_RISK_CLASSES = {"MEL", "BCC", "AKIEC"}


def skinova_pipeline(image_path, user_query, top_k=3):
    """
    Full CNN + RAG + LLM pipeline.

    1. CNN predicts HAM10000 class probabilities from the image
    2. RAG retrieves class-aware evidence for the user's query
    3. LLM writes a grounded answer; code adds safety notes + disclaimer
    """
    top_preds = predict_topk(image_path, k=3)
    predicted_class, confidence = top_preds[0]

    evidence = build_evidence_package(
        query=user_query,
        predicted_class=predicted_class,
        prediction_confidence=confidence,
        top_k=top_k,
    )

    prompt = build_skinova_prompt(evidence)
    answer = generate_response(prompt)

    # Safety notes are added in code so they never depend on the LLM.
    notes = []
    if confidence < LOW_CONFIDENCE:
        alt = ", ".join(f"{c} ({p:.0%})" for c, p in top_preds)
        notes.append(f"The classifier is not confident. Its top predictions were: {alt}.")
    if predicted_class in HIGH_RISK_CLASSES:
        notes.append("This class can be a skin cancer or pre-cancer, so please see a "
                     "dermatologist promptly rather than waiting.")
    elif any(c == "MEL" and p >= 0.15 for c, p in top_preds):
        mel_p = next(p for c, p in top_preds if c == "MEL")
        notes.append(f"Melanoma cannot be excluded (the model gave it {mel_p:.0%}). "
                     "Have this checked by a dermatologist.")

    final_answer = answer + "\n\n" + "\n".join(notes + [DISCLAIMER])

    return {
        "predicted_class": predicted_class,
        "confidence": confidence,
        "top_predictions": top_preds,
        "retrieved": [
            {k: r[k] for k in ("rank", "chunk_id", "source", "filename", "ham10000_class", "score")}
            for r in evidence["retrieval"]["results"]
        ],
        "retrieved_sources": [r["source"] for r in evidence["retrieval"]["results"]],
        "answer": final_answer,
    }


## Run the full pipeline on a test image

In [58]:
sample_image_path = "/kaggle/input/datasets/surajghuwalewala/ham1000-segmentation-and-classification/images/ISIC_0028394.jpg"

result = skinova_pipeline(
    image_path=sample_image_path,
    user_query="What is this and what should I do about it?",
)

print("=" * 80)
print("PREDICTED CLASS :", result["predicted_class"])
print("CONFIDENCE      :", f"{result['confidence']:.2%}")
print("TOP 3           :", [(c, f"{p:.1%}") for c, p in result["top_predictions"]])
print("SOURCES USED    :")
for r in result["retrieved"]:
    print(f"  {r['rank']}. {r['source']:8s} {r['ham10000_class']}  {r['chunk_id']}  score={r['score']}")
print("=" * 80)
print("ANSWER:\n")
print(result["answer"])


PREDICTED CLASS : NV
CONFIDENCE      : 99.64%
TOP 3           : [('NV', '99.6%'), ('DF', '0.2%'), ('MEL', '0.1%')]
SOURCES USED    :
  1. DermNet  NV  NV_0003  score=0.7105
  2. DermNet  NV  NV_0000  score=0.6947
  3. DermNet  NV  NV_0002  score=0.6489
ANSWER:

### AI Prediction
The image classifier predicts the presence of a Nevus (commonly known as a mole) with a high confidence level of 99.64%. However, it is important to note that this is an AI prediction and not a confirmed diagnosis. A thorough clinical assessment by a dermatologist is necessary to confirm the diagnosis.

### What the Retrieved Medical Sources Say
According to the retrieved medical sources, melanocytic naevi (moles) are common benign skin lesions due to a local proliferation of pigment cells (melanocytes). They can vary widely in appearance and may be present at birth or develop later in life. Moles can be flat or raised, and their color ranges from pink or flesh tones to dark brown, steel blue, or black.

### Re

## Save deployment artifacts

In [59]:
# Everything needed to serve Skinova outside this notebook:
#   - the Keras CNN, - the RAG folder (chunks, embeddings, FAISS index), - config.json
# The LLM itself (Qwen2.5-7B) is not re-saved: at deploy time pull it from
# Hugging Face by LLM_MODEL_ID and apply the same 4-bit config.

import os
import json
import shutil

DEPLOY_DIR = "/kaggle/working/skinova_deploy"
os.makedirs(DEPLOY_DIR, exist_ok=True)

SOURCE_MODEL = "/kaggle/input/models/eshamanohar/skinova-effiecientb0-bestmodel/keras/default/1/skinova_efficientnetb0_best.keras"
DEST_MODEL = os.path.join(DEPLOY_DIR, "skinova_efficientnetb0_best.keras")

shutil.copy2(SOURCE_MODEL, DEST_MODEL)
print("Model copied to:", DEST_MODEL)

# RAG knowledge base
shutil.copytree(
    "/kaggle/working/skinova_rag",
    os.path.join(DEPLOY_DIR, "skinova_rag"),
    dirs_exist_ok=True,
)

# Config so the serving app knows class order, preprocessing, models and thresholds
deploy_config = {
    "cnn_classes": HAM10000_CLASSES,
    "cnn_img_size": list(IMG_SIZE),
    "divide_by_255": DIVIDE_BY_255,
    "llm_model_id": LLM_MODEL_ID,
    "embedding_model": MODEL_NAME,
    "top_k_default": 3,
    "low_confidence_threshold": LOW_CONFIDENCE,
}

with open(os.path.join(DEPLOY_DIR, "config.json"), "w") as f:
    json.dump(deploy_config, f, indent=2)

print("Deployment artifacts saved to:", DEPLOY_DIR)
print(os.listdir(DEPLOY_DIR))


Model copied to: /kaggle/working/skinova_deploy/skinova_efficientnetb0_best.keras
Deployment artifacts saved to: /kaggle/working/skinova_deploy
['skinova_rag', 'skinova_efficientnetb0_best.keras', 'config.json']


In [60]:
import shutil

shutil.make_archive(
    "/kaggle/working/skinova_rag",
    "zip",
    "/kaggle/working/skinova_rag"
)

print("ZIP created successfully!")

ZIP created successfully!
